<a href="https://colab.research.google.com/github/Ravi-ranjan1801/CUDA-Lab/blob/main/cuda_lab_07_parallel_merge_sort_parallel_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%writefile parallel_merge_sort.cu
// parallel_merge_sort.cu
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>

__global__ void mergeKernel(int *input, int *output, int width, int n)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    int start = tid * 2 * width;

    if(start < n)
    {
        int mid = min(start + width, n);
        int end = min(start + 2 * width, n);

        int i = start;
        int j = mid;
        int k = start;

        while(i < mid && j < end)
        {
            if(input[i] < input[j])
                output[k++] = input[i++];
            else
                output[k++] = input[j++];
        }

        while(i < mid) output[k++] = input[i++];
        while(j < end) output[k++] = input[j++];
    }
}

void printArray(int *arr, int n)
{
    for(int i=0;i<n;i++) printf("%d ",arr[i]);
    printf("\n");
}

int main()
{
    int n;
    printf("Enter number of elements: ");
    scanf("%d",&n);

    int *h = (int*)malloc(n*sizeof(int));

    for(int i=0;i<n;i++)
        h[i] = rand()%100;

    printf("\nOriginal:\n");
    printArray(h,n);

    int *d_in,*d_out;

    cudaMalloc(&d_in,n*sizeof(int));
    cudaMalloc(&d_out,n*sizeof(int));

    cudaMemcpy(d_in,h,n*sizeof(int),cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks;

    cudaEvent_t s,e;
    cudaEventCreate(&s);
    cudaEventCreate(&e);

    cudaEventRecord(s);

    for(int width=1;width<n;width*=2)
    {
        blocks = (n + (2*width*threads -1)) / (2*width*threads);

        mergeKernel<<<blocks,threads>>>(d_in,d_out,width,n);

        int *temp = d_in;
        d_in = d_out;
        d_out = temp;
    }

    cudaEventRecord(e);
    cudaEventSynchronize(e);

    float ms;
    cudaEventElapsedTime(&ms,s,e);

    cudaMemcpy(h,d_in,n*sizeof(int),cudaMemcpyDeviceToHost);

    printf("\nSorted:\n");
    printArray(h,n);

    printf("\nGPU Time = %f ms\n",ms);

    cudaFree(d_in);
    cudaFree(d_out);
    free(h);
}

Overwriting parallel_merge_sort.cu


In [4]:
!nvcc parallel_merge_sort.cu -o sort
!./sort

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter number of elements: 10

Original:
83 86 77 15 93 35 86 92 49 21 

Sorted:
15 21 35 49 77 83 86 86 92 93 

GPU Time = 0.209408 ms
